In [1]:
import os, sys, joblib
import numpy as np, pandas as pd
from collections import Counter
sys.path.insert(0, os.path.abspath("src"))
from utils import clean_text

# --- Carregar tudo o que o Sprint 2 e 3 guardaram ---
le = joblib.load("data/splits.joblib")["label_encoder"]
preps = {"sparse": joblib.load("models/prep_sparse.joblib"),
         "dense":  joblib.load("models/prep_dense.joblib")}
F = joblib.load("models/fitted_models.joblib")
fitted, prep_kind = F["fitted"], F["prep_kind"]

# Modelo TensorFlow (opcional)
tf_model = None
try:
    import tensorflow as tf
    tf_model = tf.keras.models.load_model("models/tf_mlp.keras")
except Exception as e:
    print("TensorFlow não carregado (ignorado):", e)


def confidence(model, X):
    """Confiança da classe prevista (probabilidade ou softmax das margens)."""
    if hasattr(model, "predict_proba"):
        return float(model.predict_proba(X).max())
    if hasattr(model, "decision_function"):
        d = model.decision_function(X)[0]
        e = np.exp(d - d.max())
        return float((e / e.sum()).max())
    return float("nan")


def prever(texto):
    t = clean_text(texto)
    if not t:
        print("Texto vazio depois da limpeza. Escreve algo com letras.")
        return
    X = {k: p.transform([t]) for k, p in preps.items()}   # transforma uma vez por tipo de features
    rows = []
    for name, model in fitted.items():
        Xm = X[prep_kind[name]]
        lang = le.classes_[model.predict(Xm)[0]]
        rows.append({"modelo": name, "língua": lang, "confiança": round(confidence(model, Xm), 3)})
    if tf_model is not None:
        p = tf_model.predict(X["sparse"].astype("float32").toarray(), verbose=0)[0]
        rows.append({"modelo": "TensorFlow MLP", "língua": le.classes_[p.argmax()], "confiança": round(float(p.max()), 3)})

    df_res = pd.DataFrame(rows).sort_values("confiança", ascending=False).reset_index(drop=True)
    votos = Counter(df_res["língua"]).most_common(1)[0]
    display(df_res)
    print(f"🗳️  Voto da maioria: {votos[0]} ({votos[1]}/{len(df_res)} modelos)")


# --- Loop interativo ---
while True:
    texto = input("\nEscreve um texto (ou 'sair'): ").strip()
    if texto.lower() == "sair":
        break
    prever(texto)

TensorFlow não carregado (ignorado): No module named 'tensorflow'


,modelo,língua,confiança
0,MultinomialNB,Portuguese,1.000
1,BernoulliNB,Chinese,1.000
2,kNN (cosine),Portuguese,1.000
3,GradientBoosting (Hist),Portuguese,0.999
4,XGBoost,Portuguese,0.997
5,GaussianNB,Latin,0.986
6,DecisionTree,Spanish,0.750
7,SVC RBF (SVM não-linear),Portuguese,0.632
8,LogisticRegression,Portuguese,0.619
9,RandomForest (bagging),Portuguese,0.165


🗳️  Voto da maioria: Portuguese (8/12 modelos)


,modelo,língua,confiança
0,BernoulliNB,Chinese,1.000
1,GaussianNB,Latin,1.000
2,MultinomialNB,Portuguese,0.978
3,SVC RBF (SVM não-linear),Portuguese,0.633
4,GradientBoosting (Hist),Estonian,0.476
5,kNN (cosine),Estonian,0.400
6,RandomForest (bagging),Japanese,0.275
7,XGBoost,Estonian,0.274
8,DecisionTree,Estonian,0.194
9,LogisticRegression,Estonian,0.184


🗳️  Voto da maioria: Estonian (6/12 modelos)


,modelo,língua,confiança
0,BernoulliNB,Chinese,1.000
1,GaussianNB,Latin,1.000
2,MultinomialNB,Indonesian,0.972
3,GradientBoosting (Hist),Indonesian,0.820
4,SVC RBF (SVM não-linear),Portuguese,0.632
5,XGBoost,Indonesian,0.627
6,LogisticRegression,Indonesian,0.433
7,kNN (cosine),Indonesian,0.400
8,DecisionTree,Estonian,0.194
9,LinearSVC (SVM linear),English,0.127


🗳️  Voto da maioria: Indonesian (5/12 modelos)
